# Analyze Local Concept Statistics

This notebook computes validation-set statistics for a trained local concept model and compares prediction vs target distributions.

Metrics:
- concept-wise mean probability
- concept-wise top-1 winner ratio
- concept-wise positive pixel ratio

Both prediction-side and target-side statistics are reported.

In [4]:
from __future__ import annotations

import sys
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from configs.defaults import build_videomae_args
from datasets.local_video_dataset import LocalConceptVideoDataset
from models.backbone import FrozenVideoMAEBackbone
from models.localizer import VideoMAELocalizer

In [5]:
# Configuration
CONFIG = {
    'anno_path': Path('/data/lwi2765/repos/XAI/Video_Language_XAI/dataset/KTH/val.csv'),
    'data_root': Path('/local_datasets/kth/video'),
    'pseudo_mask_root': Path('/data/dataset/VideoXAI/optical_flow/kth_flow_21'),
    'localizer_ckpt': REPO_ROOT / 'runs' / 'KTH_21_no_bias' / 'best.pt',
    'backbone': 'vmae_vit_base_patch16_224',
    'finetune': '/data/lwi2765/repos/VideoMAE/videoMAE/result/KTH/OUT/KTH_videomae_finetune.pth',
    'data_set': 'kth',
    'nb_classes': 6,
    'num_concepts': 21,
    'batch_size': 4,
    'num_workers': 4,
    'device': 'cuda',
    'block_index': 6,
    'num_frames': 16,
    'num_segments': 1,
    'sampling_rate': 4,
    'tubelet_size': 2,
    'input_size': 224,
    'patch_size': 16,
    'fc_drop_rate': 0.0,
    'drop': 0.0,
    'drop_path': 0.1,
    'attn_drop_rate': 0.0,
    'use_checkpoint': False,
    'use_mean_pooling': True,
    'init_scale': 0.001,
    'model_key': 'model|module',
    'model_prefix': '',
    'deterministic_spatial': True,
    'view_mode': 'center_uniform',
    'target_cache_root': None,
    'threshold': 0.1,
}
CONFIG

{'anno_path': PosixPath('/data/lwi2765/repos/XAI/Video_Language_XAI/dataset/KTH/val.csv'),
 'data_root': PosixPath('/local_datasets/kth/video'),
 'pseudo_mask_root': PosixPath('/data/dataset/VideoXAI/optical_flow/kth_flow_21'),
 'localizer_ckpt': PosixPath('/data/lwi2765/repos/XAI/Video_Language_XAI/CBM_training_ver2/runs/KTH_21_no_bias/best.pt'),
 'backbone': 'vmae_vit_base_patch16_224',
 'finetune': '/data/lwi2765/repos/VideoMAE/videoMAE/result/KTH/OUT/KTH_videomae_finetune.pth',
 'data_set': 'kth',
 'nb_classes': 6,
 'num_concepts': 21,
 'batch_size': 4,
 'num_workers': 4,
 'device': 'cuda',
 'block_index': 6,
 'num_frames': 16,
 'num_segments': 1,
 'sampling_rate': 4,
 'tubelet_size': 2,
 'input_size': 224,
 'patch_size': 16,
 'fc_drop_rate': 0.0,
 'drop': 0.0,
 'drop_path': 0.1,
 'attn_drop_rate': 0.0,
 'use_checkpoint': False,
 'use_mean_pooling': True,
 'init_scale': 0.001,
 'model_key': 'model|module',
 'model_prefix': '',
 'deterministic_spatial': True,
 'view_mode': 'center_u

In [6]:
device = torch.device(CONFIG['device'])
args = SimpleNamespace(**CONFIG)
model_args = build_videomae_args(args)

dataset = LocalConceptVideoDataset(
    anno_path=CONFIG['anno_path'],
    data_root=CONFIG['data_root'],
    data_set=CONFIG['data_set'],
    pseudo_mask_root=CONFIG['pseudo_mask_root'],
    tubelet_size=CONFIG['tubelet_size'],
    patch_size=CONFIG['patch_size'],
    target_cache_root=CONFIG['target_cache_root'],
    num_frames=CONFIG['num_frames'],
    sampling_rate=CONFIG['sampling_rate'],
    input_size=CONFIG['input_size'],
    deterministic=CONFIG['deterministic_spatial'],
    view_mode=CONFIG['view_mode'],
)

def collate_batch(batch):
    inputs = torch.stack([item[0] for item in batch], dim=0)
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    targets = torch.stack([item[2] for item in batch], dim=0)
    metas = [item[3] for item in batch]
    return inputs, labels, targets, metas

loader = DataLoader(
    dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    collate_fn=collate_batch,
)

backbone = FrozenVideoMAEBackbone.from_args(model_args, device=device)
model = VideoMAELocalizer(backbone, out_channels=CONFIG['num_concepts']).to(device)
ckpt = torch.load(CONFIG['localizer_ckpt'], map_location='cpu')
model.load_state_dict(ckpt['model'], strict=True)
model.eval()

len(dataset)

/data/lwi2765/repos/XAI/PCBEAR/CBM_training/data_utils.py:245: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(args.finetune, map_location='cpu')


Load ckpt from /data/lwi2765/repos/VideoMAE/videoMAE/result/KTH/OUT/KTH_videomae_finetune.pth
Load state_dict by model_key = model



*********** VMAE Load ***************


/tmp/ipykernel_2572435/4133917149.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CONFIG['localizer_ckpt'], map_location='cpu')


863

In [7]:
num_concepts = CONFIG['num_concepts']
threshold = CONFIG['threshold']

pred_prob_sum = torch.zeros(num_concepts, dtype=torch.float64)
pred_positive_count = torch.zeros(num_concepts, dtype=torch.float64)
pred_top1_wins = torch.zeros(num_concepts, dtype=torch.float64)

target_prob_sum = torch.zeros(num_concepts, dtype=torch.float64)
target_positive_count = torch.zeros(num_concepts, dtype=torch.float64)
target_top1_wins = torch.zeros(num_concepts, dtype=torch.float64)

concept_pixel_count = torch.zeros(num_concepts, dtype=torch.float64)
total_pixels = 0.0

with torch.no_grad():
    for videos, _labels, targets, _metas in tqdm(loader, desc='analyze'):
        videos = videos.to(device, non_blocking=True)
        logits, _ = model(videos)
        probs = torch.sigmoid(logits).cpu()
        preds = (probs >= threshold).to(torch.float32)
        targets = targets.cpu().to(torch.float32)

        batch_pixels_per_concept = probs.shape[0] * probs.shape[2] * probs.shape[3] * probs.shape[4]
        concept_pixel_count += torch.full((num_concepts,), batch_pixels_per_concept, dtype=torch.float64)
        total_pixels += batch_pixels_per_concept

        pred_prob_sum += probs.sum(dim=(0, 2, 3, 4), dtype=torch.float64)
        pred_positive_count += preds.sum(dim=(0, 2, 3, 4), dtype=torch.float64)
        pred_winners = probs.argmax(dim=1)
        for concept_idx in range(num_concepts):
            pred_top1_wins[concept_idx] += (pred_winners == concept_idx).sum().item()

        target_prob_sum += targets.sum(dim=(0, 2, 3, 4), dtype=torch.float64)
        target_positive_count += (targets > 0).sum(dim=(0, 2, 3, 4), dtype=torch.float64)
        target_winners = targets.argmax(dim=1)
        active_target = targets.max(dim=1).values > 0
        for concept_idx in range(num_concepts):
            target_top1_wins[concept_idx] += ((target_winners == concept_idx) & active_target).sum().item()

pred_mean_prob = pred_prob_sum / concept_pixel_count.clamp_min(1.0)
pred_positive_ratio = pred_positive_count / concept_pixel_count.clamp_min(1.0)
pred_top1_ratio = pred_top1_wins / max(total_pixels, 1.0)

target_mean_prob = target_prob_sum / concept_pixel_count.clamp_min(1.0)
target_positive_ratio = target_positive_count / concept_pixel_count.clamp_min(1.0)
target_top1_ratio = target_top1_wins / max(total_pixels, 1.0)

stats_df = pd.DataFrame({
    'concept_idx': list(range(num_concepts)),
    'pred_mean_prob': pred_mean_prob.numpy(),
    'pred_top1_winner_ratio': pred_top1_ratio.numpy(),
    'pred_positive_pixel_ratio': pred_positive_ratio.numpy(),
    'target_mean_prob': target_mean_prob.numpy(),
    'target_top1_winner_ratio': target_top1_ratio.numpy(),
    'target_positive_pixel_ratio': target_positive_ratio.numpy(),
    'top1_gap_pred_minus_target': (pred_top1_ratio - target_top1_ratio).numpy(),
    'positive_gap_pred_minus_target': (pred_positive_ratio - target_positive_ratio).numpy(),
}).sort_values('pred_top1_winner_ratio', ascending=False)

stats_df.head(10)

analyze:   0%|          | 0/216 [00:00<?, ?it/s]

,concept_idx,pred_mean_prob,pred_top1_winner_ratio,pred_positive_pixel_ratio,target_mean_prob,target_top1_winner_ratio,target_positive_pixel_ratio,top1_gap_pred_minus_target,positive_gap_pred_minus_target
7,7,0.008384,0.581979,0.023026,0.008079,0.006918,0.008079,0.575060,0.014947
5,5,0.005920,0.291406,0.016285,0.005357,0.004551,0.005357,0.286855,0.010928
14,14,0.000542,0.044591,0.000416,0.001204,0.000648,0.001204,0.043943,-0.000788
15,15,0.001337,0.030793,0.002807,0.001414,0.001181,0.001414,0.029612,0.001392
1,1,0.001136,0.010512,0.001134,0.001392,0.001385,0.001392,0.009127,-0.000258
9,9,0.002021,0.009304,0.005030,0.002449,0.001909,0.002449,0.007395,0.002581
4,4,0.002380,0.008113,0.005794,0.002551,0.002178,0.002551,0.005936,0.003243
12,12,0.001203,0.005895,0.002646,0.001238,0.001045,0.001238,0.004850,0.001408
2,2,0.001913,0.005134,0.003425,0.002503,0.002190,0.002503,0.002943,0.000922
8,8,0.002626,0.004662,0.006635,0.002867,0.001493,0.002867,0.003169,0.003767


In [8]:
stats_df

,concept_idx,pred_mean_prob,pred_top1_winner_ratio,pred_positive_pixel_ratio,target_mean_prob,target_top1_winner_ratio,target_positive_pixel_ratio,top1_gap_pred_minus_target,positive_gap_pred_minus_target
7,7,8.384279e-03,0.581979,0.023026,0.008079,0.006918,0.008079,0.575060,0.014947
5,5,5.920488e-03,0.291406,0.016285,0.005357,0.004551,0.005357,0.286855,0.010928
14,14,5.419162e-04,0.044591,0.000416,0.001204,0.000648,0.001204,0.043943,-0.000788
15,15,1.337268e-03,0.030793,0.002807,0.001414,0.001181,0.001414,0.029612,0.001392
1,1,1.135631e-03,0.010512,0.001134,0.001392,0.001385,0.001392,0.009127,-0.000258
9,9,2.020577e-03,0.009304,0.005030,0.002449,0.001909,0.002449,0.007395,0.002581
4,4,2.380127e-03,0.008113,0.005794,0.002551,0.002178,0.002551,0.005936,0.003243
12,12,1.203264e-03,0.005895,0.002646,0.001238,0.001045,0.001238,0.004850,0.001408
2,2,1.913352e-03,0.005134,0.003425,0.002503,0.002190,0.002503,0.002943,0.000922
8,8,2.626429e-03,0.004662,0.006635,0.002867,0.001493,0.002867,0.003169,0.003767


In [9]:
# Optional: save to CSV
output_csv = REPO_ROOT / 'runs' / 'KTH_21_ddp' / 'concept_stats_val_with_target.csv'
stats_df.to_csv(output_csv, index=False)
output_csv

PosixPath('/data/lwi2765/repos/XAI/Video_Language_XAI/CBM_training_ver2/runs/KTH_21_ddp/concept_stats_val_with_target.csv')